# RUNG 13–16 — death wave · arena · atlas map · neoantigen burden (CPU; Census + MHCflurry optional)

**Run all** (cells 1–6 CPU ~3 min) — Census (cell 7) and MHCflurry burden (cell 8) are optional/heavier.

1. **RUNG-13** (`scripts/38`) — single-cell EARM switch → coupled **death wave**.
2. **RUNG-14 arena** (`scripts/39`) — **9 + 2** strategies × 12 regimes → leaderboard (HITS/CLOSE/FAR). incl. `ferroptosis_wave`, `synthetic_lethal`.
3. **RUNG-15 map** (`scripts/40 map`) — which cancers each strategy reaches on the neoantigen axis.
4. **RUNG-15 census** (`scripts/40 census`, cell 7, Census) — surface-antigen q_t/q_n → mechanism map. *Found: 0/25 single surface markers safe (anti-correlation).*
5. **RUNG-16 burden** (`scripts/41`, cell 8, MHCflurry) — with the **full clonal mutation repertoire**, what fraction of patients have ≥1 clean handle to **seed the wave**? (high-TMB cancers ≫ shared hotspots).

**Ceiling:** coupling/delivery/TCR = wet-lab residual; this bounds *in-silico addressability/containment*, not cures.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log
from scripts.runlog import new_log
RUNLOG = new_log('rung13_16_arena', repo_dir='.')
print('[CELL 2] ✓ logging to', RUNLOG)

In [ ]:
#@title Cell 3 — VALIDATE ports (fast). Stop if anything fails.
import sys
from scripts.runlog import run_logged
rcs = [run_logged([sys.executable,'-u',f'scripts/{s}','selftest'], RUNLOG) for s in
       ('38_apoptosis_wave.py','39_mechanism_arena.py','40_atlas_mechanism_map.py','41_neoantigen_burden_coverage.py')]
print('[CELL 3]', '✓ all validated' if all(r==0 for r in rcs) else f'✗ FAILED {rcs} — stop here')

In [ ]:
#@title Cell 4 — RUNG-13: death wave
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/38_apoptosis_wave.py','run'], RUNLOG)
from IPython.display import Image, display
if os.path.exists('runs/rung13_apoptosis/rung13_apoptosis.png'): display(Image('runs/rung13_apoptosis/rung13_apoptosis.png'))
if os.path.exists('runs/rung13_apoptosis/rung13_apoptosis.json'):
    print('DECISIVE:', json.load(open('runs/rung13_apoptosis/rung13_apoptosis.json'))['DECISIVE'])

In [ ]:
#@title Cell 5 — RUNG-14: THE ARENA. Watch the LEADERBOARD.
mode = 'run' #@param ['run', 'quick']
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/39_mechanism_arena.py', mode], RUNLOG)
from IPython.display import Image, display
if os.path.exists('runs/rung14_arena/rung14_arena.png'): display(Image('runs/rung14_arena/rung14_arena.png'))
if os.path.exists('runs/rung14_arena/rung14_arena.json'):
    d=json.load(open('runs/rung14_arena/rung14_arena.json'))
    print('LEADERBOARD:', ' > '.join(f"{b['mechanism']}({b['safe_fraction']:.2f})" for b in d['leaderboard_rigorous']))
    print('\nDECISIVE:', d['DECISIVE'])

In [ ]:
#@title Cell 6 — RUNG-15 MAP: which cancers each strategy addresses (neoantigen)
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/40_atlas_mechanism_map.py','map'], RUNLOG)
from IPython.display import Image, display
if os.path.exists('runs/rung15_atlas_map/rung15_neoantigen_map.png'): display(Image('runs/rung15_atlas_map/rung15_neoantigen_map.png'))
if os.path.exists('runs/rung15_atlas_map/rung15_neoantigen_map.json'):
    print('DECISIVE:', json.load(open('runs/rung15_atlas_map/rung15_neoantigen_map.json'))['DECISIVE'])

In [ ]:
#@title Cell 7 — (OPTIONAL, Census) RUNG-15 surface-antigen map  [~15 min]
#@markdown Needs `cellxgene_census`; resumable tiles. q_t needs the RUNG-5 tumour cache on Drive (else q_n-only).
import os, sys, importlib.util
from scripts.runlog import run_logged
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd='/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG12P_CACHE']=f'{cd}/rung15_tiles'; os.makedirs(os.environ['RUNG12P_CACHE'], exist_ok=True)
    os.environ['LOGICGATE_CACHE']=f'{cd}/rung5_cache.npz'
    print('[CELL 7] RUNG-5 tumour cache:', 'FOUND' if os.path.exists(f'{cd}/rung5_cache.r5.tumour.npz') else 'absent (q_n-only)')
except Exception as e:
    print('[CELL 7] no Drive (', type(e).__name__, ')')
for pk,pn in [('cellxgene_census','cellxgene-census'),('scanpy','scanpy')]:
    if importlib.util.find_spec(pk) is None:
        run_logged([sys.executable,'-m','pip','install','-q',pn], RUNLOG, label=f'pip {pn}')
rc = run_logged([sys.executable,'-u','scripts/40_atlas_mechanism_map.py','census'], RUNLOG)
import json
if os.path.exists('runs/rung15_atlas_map/rung15_surface_map.json'):
    print('DECISIVE:', json.load(open('runs/rung15_atlas_map/rung15_surface_map.json'))['DECISIVE'])
print('[CELL 7]', '✓ done' if rc==0 else f'(exit {rc}; re-run to resume, or skip)')

In [ ]:
#@title Cell 8 — (OPTIONAL, MHCflurry) RUNG-16 clonal neoantigen burden  [~5 min + model dl]
#@markdown Installs MHCflurry + fetches presentation models, then per-patient ≥1-clean-handle coverage per cancer.
import sys, os, importlib.util
from scripts.runlog import run_logged
if importlib.util.find_spec('mhcflurry') is None:
    run_logged([sys.executable,'-m','pip','install','-q','mhcflurry'], RUNLOG, label='pip mhcflurry')
    run_logged(['mhcflurry-downloads','fetch','models_class1_presentation'], RUNLOG, label='fetch models')
rc = run_logged([sys.executable,'-u','scripts/41_neoantigen_burden_coverage.py'], RUNLOG)
from IPython.display import Image, display
if os.path.exists('runs/rung16_burden/rung16_burden.png'): display(Image('runs/rung16_burden/rung16_burden.png'))
import json
if os.path.exists('runs/rung16_burden/rung16_burden.json'):
    print('DECISIVE:', json.load(open('runs/rung16_burden/rung16_burden.json'))['DECISIVE'])
print('[CELL 8]', '✓ done' if rc==0 else f'(exit {rc})')

In [ ]:
#@title Cell 9 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung13_16_arena_','').replace('.log','')
b = f'/content/rung13_16_run_{stem}.zip'
ps = sorted(set(sum([glob.glob(f'runs/{d}/*.png')+glob.glob(f'runs/{d}/*.json')
                     for d in ('rung13_apoptosis','rung14_arena','rung15_atlas_map','rung16_burden')], [])+[str(RUNLOG)]))
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 9] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')